# TERRA - DS Part 2 - Data Cleaning
**Sources:** Open-Meteo, Eurostat, World Bank API

**Year range:** 2010–2023

**Output:** Merged table `../datasets/processed/merged_data.csv`


In [4]:
import pandas as pd
import os
from functools import reduce

wb = pd.read_csv("../datasets/raw/worldbank_raw.csv")
eu = pd.read_csv("../datasets/raw/eurostat_raw.csv")
open_meteo = pd.read_csv("../datasets/raw/open_meteo_raw.csv")

wb['year'] = wb['year'].astype(int)
wb['country_code'] = wb['country_code'].astype('category')

eu['year'] = eu['year'].astype(int)
eu['country_code'] = eu['country_code'].astype('category')
open_meteo['year'] = open_meteo['year'].astype(int)
open_meteo['country_code'] = open_meteo['country_code'].astype('category')

wb = wb.drop_duplicates(subset=['country_code', 'year'])
eu = eu.drop_duplicates(subset=['country_code', 'year'])
open_meteo = open_meteo.drop_duplicates(subset=['country_code', 'year'])

numeric_cols = ['gdp_per_capita', 'unemployment_rate', 'population', 'urban_pct']
wb[numeric_cols] = wb[numeric_cols].apply(pd.to_numeric, errors='coerce')
eu['asylum_applications'] = pd.to_numeric(eu['asylum_applications'], errors='coerce')
open_meteo[['temp_mean', 'heatwave_days', 'precip_total', 'precip_days_heavy', 'dry_days', 'evapotrans_total']] = open_meteo[['temp_mean', 'heatwave_days', 'precip_total', 'precip_days_heavy', 'dry_days', 'evapotrans_total']].apply(pd.to_numeric, errors='coerce')

wb.isna().sum()
eu.isna().sum()
open_meteo.isna().sum()

eu['asylum_applications'] = pd.to_numeric(eu['asylum_applications'], errors='coerce')
open_meteo[['temp_mean', 'heatwave_days', 'precip_total', 'precip_days_heavy', 'dry_days', 'evapotrans_total']] = open_meteo[['temp_mean', 'heatwave_days', 'precip_total', 'precip_days_heavy', 'dry_days', 'evapotrans_total']].apply(pd.to_numeric, errors='coerce')

eu = eu.drop_duplicates(subset=['country_code', 'year'])
open_meteo = open_meteo.drop_duplicates(subset=['country_code', 'year'])

eu['asylum_applications'] = eu['asylum_applications'].fillna(0)
open_meteo[['temp_mean', 'heatwave_days', 'precip_total', 'precip_days_heavy', 'dry_days', 'evapotrans_total']] = open_meteo[['temp_mean', 'heatwave_days', 'precip_total', 'precip_days_heavy', 'dry_days', 'evapotrans_total']].fillna(0)

df = wb.merge(eu, on=['country_code', 'year'], how='left')
df = df.merge(open_meteo, on=['country_code', 'year'], how='left')

df.isna().sum()
df = df.sort_values(['country_code', 'year']).reset_index(drop=True)
print(f"Final shape: {df.shape}")
df.head()

Final shape: (378, 13)


,year,gdp_per_capita,unemployment_rate,population,urban_pct,country_code,asylum_applications,temp_mean,heatwave_days,precip_total,precip_days_heavy,dry_days,evapotrans_total
0,2010,46611.139342,4.883,8363404.0,67.104743,AT,11060.0,9.422351,0,783.7,7,229,759.99400
1,2011,51116.895352,4.637,8391643.0,67.148426,AT,14455.0,10.900239,0,498.1,2,277,873.22080
2,2012,48250.405914,4.909,8429991.0,67.208591,AT,17450.0,11.087264,0,552.6,1,254,891.62646
3,2013,50305.354577,5.367,8479823.0,67.298131,AT,17520.0,10.479536,2,709.0,4,237,810.21140
4,2014,51314.972262,5.674,8546356.0,67.415433,AT,28065.0,11.703919,0,796.7,4,237,811.51587


In [5]:
df.to_csv("../datasets/processed/merged_data.csv", index=False)
print("Merged dataset saved to ../datasets/processed/merged_data.csv")

Merged dataset saved to ../datasets/processed/merged_data.csv
